# Medical Insurance Cost Analysis & Predictive Modelling
**Project 2 — Full Data Analytics Pipeline**
**Author:** Matile Durin Mohwaduba

---

## Business Problem

Medical insurance pricing is not random. Insurers use a combination of demographic and lifestyle factors to estimate how much a customer is likely to cost them in healthcare claims — and price their premiums accordingly.

This project analyses a real-world medical insurance dataset to answer two connected business questions:

> **1. Which factors most significantly drive the cost of medical insurance charges?**
> **2. Can we build a model that accurately predicts what a new customer's insurance charges will be, based on their profile?**

Answering these questions has direct financial value:
- **For insurers:** More accurate pricing means fewer underpriced high-risk customers
- **For policymakers:** Understanding cost drivers enables better public health interventions
- **For individuals:** Knowing which habits (like smoking) compound financial risk creates actionable incentive

### Dataset Overview
The dataset is sourced from a widely-used machine learning reference dataset containing **1,338 medical insurance records** with the following features:

| Column | Description |
|--------|-------------|
| `age` | Age of the primary insurance beneficiary |
| `sex` | Gender of the beneficiary |
| `bmi` | Body Mass Index — a measure of body fat based on height and weight |
| `children` | Number of dependents covered under the policy |
| `smoker` | Whether the beneficiary smokes (yes/no) |
| `region` | Geographic region in the USA |
| `charges` | Actual annual medical insurance charges billed (target variable) |

### Analytical Pipeline

| Step | Task | Status |
|------|------|--------|
| 1 | Data Collection & Dataset Understanding | Complete |
| 2 | Data Cleaning & Preprocessing | Complete |
| 3 | Exploratory Data Analysis (EDA) | Complete |
| 4 | Data Visualisation | Complete |
| 5 | Predictive Modelling | Complete |

---

## Step 1: Data Collection & Dataset Understanding

### Goal
Load the dataset and understand its structure — column names, data types, dimensions, and what each feature represents.

### Why load from a URL?
Loading directly from a public GitHub URL ensures full reproducibility. Anyone running this notebook will work with the identical source data regardless of their local environment — a standard best practice in data analytics and collaborative projects.

### Libraries
- `pandas` — Python's primary data manipulation library, used throughout this project for loading, cleaning, transforming, and summarising data.

In [ ]:
print("Hello World - Matile Durin's Project 2 Initiated")

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(url)

print("Financial data loaded successfully!")

### Dataset Dimensions & Structure

In [ ]:
print("Dataset Shape (Rows, Columns):", df.shape)
print("\n--- Detailed Column Breakdown ---")
df.info()

**Observations:**
- The dataset contains **1,338 rows and 7 columns** — a compact, well-structured dataset with no immediately obvious structural problems
- Three columns are numeric (`age`, `bmi`, `charges`), one is integer (`children`), and three are categorical strings (`sex`, `smoker`, `region`)
- The target variable `charges` is correctly stored as float64 — ready for regression modelling
- No null counts are reported by `.info()`, but we will formally verify this in the cleaning step

**Key question this dataset can answer:** Given a person's age, BMI, smoking status, and family situation — how much should their annual insurance cost?

---

## Step 2: Data Cleaning & Preprocessing

### Goal
Prepare the dataset for analysis by identifying and resolving data quality issues — duplicates, missing values, structural errors, and logical impossibilities.

### Why cleaning comes before analysis
Analysing dirty data produces unreliable conclusions. Even one duplicate record or one impossible value can distort averages, skew model training, and lead to decisions based on fiction rather than fact. The principle is simple: **garbage in, garbage out.**

We check for three categories of issues:
1. **Exact duplicate rows** — identical records that would artificially inflate counts
2. **Missing values** — blank or null cells that break calculations
3. **Structural and logical errors** — typos in categories, or values that are impossible in the real world (e.g. negative age)

### 2.1 Duplicate Check & Removal

In [ ]:
# 1. Check for any exact duplicate patient records
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")

# 2. Remove those duplicates to ensure accuracy
df_cleaned = df.drop_duplicates()
print(f"Cleaned dataset shape: {df_cleaned.shape}")

# 3. Save this pristine version as a new CSV, just like the Telco project
df_cleaned.to_csv('Cleaned_Medical_Insurance.csv', index=False)
print("Cleaned dataset saved successfully as 'Cleaned_Medical_Insurance.csv'!")

**Observation:** One duplicate row was found and removed, leaving **1,337 clean records**. While a single duplicate seems minor, in a dataset of this size it represents 0.07% of the data — and in a predictive model, even small distortions in training data can affect accuracy. Removing it is the correct decision.

The cleaned dataset is immediately saved as a CSV for use in all subsequent steps.

### 2.2 Missing Values & Data Integrity Check

In [ ]:
# 1. Formal Check for Missing Values (Blank cells)
print("--- Missing Values Check ---")
print(df_cleaned.isnull().sum())

# 2. Check for Structural Errors (Typos in text columns)
print("\n--- Unique Categories Check ---")
print("Regions found:", df_cleaned['region'].unique())
print("Sex categories found:", df_cleaned['sex'].unique())
print("Smoker categories found:", df_cleaned['smoker'].unique())

# 3. Check for Logical Errors (Impossible real-world numbers)
print("\n--- Logical Errors Check ---")
print("Youngest person (Minimum Age):", df_cleaned['age'].min())
print("Lowest insurance charge:", df_cleaned['charges'].min())

**Observations — all checks passed:**

- **Missing values:** Zero nulls across all 7 columns — the dataset is complete
- **Structural integrity:** All categorical columns contain only the expected values:
  - `region`: four US regions (northeast, northwest, southeast, southwest) — no typos
  - `sex`: male/female only — no mixed casing or spelling variants
  - `smoker`: yes/no only — clean binary encoding
- **Logical integrity:** Minimum age is 18 (no child records in an adult insurance dataset) and minimum charge is above zero (no impossible negative billing amounts)

**Cleaning summary:** The dataset required minimal cleaning — one duplicate removed, all other checks passed. This is relatively rare in real-world data work and reflects a well-maintained source dataset.

The data is now ready for analysis.

---

## Step 3: Exploratory Data Analysis (EDA)

### Goal
Analyse the dataset to discover patterns, trends, and relationships between variables — particularly what drives insurance charges higher or lower.

### Why EDA before modelling?
A machine learning model will find patterns whether they are meaningful or not. EDA ensures we understand the data well enough to:
- Identify the most important features before the model does
- Catch any remaining data quality issues (outliers, skewed distributions)
- Form hypotheses that the visualisation and modelling steps will confirm or challenge

We focus on two high-value questions:
1. What is the **financial cost of smoking** in insurance terms?
2. How does **age** compound insurance charges over a lifetime?

In [ ]:
# Re-load the clean data just to be safe
df_cleaned = pd.read_csv('Cleaned_Medical_Insurance.csv')

print("--- The Financial Cost of Smoking ---")
# This calculates the average insurance charge based on whether they smoke or not
smoking_costs = df_cleaned.groupby('smoker')['charges'].mean().round(2)
print(smoking_costs)

print("\n--- The Financial Impact of Age (Sample) ---")
# Let's look at the average cost for a 20-year-old vs a 50-year-old
age_costs = df_cleaned[df_cleaned['age'].isin([20, 50])].groupby('age')['charges'].mean().round(2)
print(age_costs)

**Observations — striking findings:**

**Smoking premium:** Non-smokers pay an average of approximately **$8,440/year** in insurance charges. Smokers pay approximately **$32,050/year** — nearly **4 times more**. This is the single largest cost driver in the dataset by a significant margin. The financial penalty for smoking is not a small surcharge; it is a fundamentally different pricing tier.

**Age compounding:** Average charges for 50-year-olds are substantially higher than for 20-year-olds, confirming that age compounds medical risk over time. However, age alone is not the full story — as the visualisation step will show, age's impact is dramatically amplified by smoking status.

These two findings form the analytical core of this project.

---

## Step 4: Data Visualisation

### Goal
Translate the numerical findings from EDA into clear, professional visual representations that communicate insights at a glance.

### Chart design decisions
Two charts are produced side by side, each answering a different dimension of the same question:

- **Chart 1 (Bar chart):** Compares average insurance cost by smoker status — the clearest way to show a two-category comparison
- **Chart 2 (Scatter plot):** Maps every individual patient by age and charges, coloured by smoker status — this reveals not just that smokers pay more, but *how that gap widens with age*

**Color logic:** Red for smokers (elevated risk), green for non-smokers (lower risk). This is semantically meaningful — the colors carry the message before the reader reads the labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Load the clean data
df_cleaned = pd.read_csv('Cleaned_Medical_Insurance.csv')

# Set a pristine, professional visual theme
sns.set_theme(style="white", context="notebook")

# Define our risk colors: Red for Yes (Smoker), Green for No (Non-Smoker)
risk_colors = {'yes': '#e74c3c', 'no': '#2ecc71'}

# Create a wide canvas for two side-by-side charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: The Bar Chart (Updated to remove the FutureWarning)
sns.barplot(data=df_cleaned, x='smoker', y='charges', hue='smoker', ax=axes[0], palette=risk_colors, errorbar=None, legend=False)
axes[0].set_title('Average Insurance Cost by Habit', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Smoker Status', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average Annual Charges ($)', fontsize=12, fontweight='bold')
sns.despine(ax=axes[0])

# Chart 2: The Scatter Plot (Lifetime Risk Mapping)
sns.scatterplot(data=df_cleaned, x='age', y='charges', hue='smoker', ax=axes[1], palette=risk_colors, alpha=0.7, s=60)
axes[1].set_title('How Age and Habits Compound Financial Risk', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Age of Patient', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Total Charges ($)', fontsize=12, fontweight='bold')
axes[1].legend(title='Smoker', bbox_to_anchor=(1.05, 1), loc='upper left')
sns.despine(ax=axes[1])

# Neatly format the layout and display
plt.tight_layout()
plt.show()

**Observations — what the charts reveal:**

**Bar chart:** The cost gap between smokers and non-smokers is visually dramatic. This is not a marginal difference — the bars are in completely different height categories. The chart communicates the finding instantly without requiring the reader to process numbers.

**Scatter plot — the more important chart:** Three distinct pricing bands are visible:
- **Lower band (green):** Non-smokers of all ages — charges rise gradually with age but remain relatively contained
- **Middle band (green, upper):** Older non-smokers with other risk factors (likely high BMI)
- **Upper band (red):** Smokers — charges start high in young adulthood and compound dramatically with age

The scatter plot reveals something the bar chart cannot: **smoking does not just add a flat premium — it interacts with age to produce exponentially higher charges over a lifetime.** A 60-year-old smoker faces charges that dwarf those of a 60-year-old non-smoker by a factor that far exceeds the average difference shown in the bar chart.

**Business implication:** Insurance companies are not overcharging smokers arbitrarily — the data shows the actual cost difference is real and substantial. From a public health perspective, helping customers quit smoking is also the single highest-impact financial intervention available.

---

## Step 5: Predictive Modelling

### Goal
Build a machine learning model that can predict a new customer's insurance charges based on their profile — turning our analytical findings into a working predictive tool.

### Model selection: Random Forest Regressor
We use a **Random Forest Regressor** for the following reasons:
- It handles a mix of numeric and categorical features well
- It is robust to outliers (which exist in the charges column for high-risk smokers)
- It captures non-linear relationships — such as the age-smoking interaction we observed in EDA
- It provides feature importance scores, telling us which variables the model relies on most

### Pipeline overview
1. **Encode categoricals** — convert text columns (sex, smoker, region) into numbers using `get_dummies`, since machine learning models require numeric input
2. **Define X and y** — separate the features (input variables) from the target (charges)
3. **Train-test split** — reserve 20% of data for testing; the model never sees this during training
4. **Train the model** — fit the Random Forest on the 80% training data
5. **Evaluate** — measure performance on the unseen 20% test data using R² and MAE

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Prepare the data: The AI only understands numbers, so we convert text (like 'yes'/'no') into 1s and 0s
df_encoded = pd.get_dummies(df_cleaned, drop_first=True)

# 2. Separate our target (the money) from our clues (the human habits)
y = df_encoded['charges']
X = df_encoded.drop('charges', axis=1)

# 3. Split the data: 80% for the AI to learn, 20% to test its intuition
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize and train the Machine Learning Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Make financial predictions on the unseen test patients
predictions = model.predict(X_test)

# 6. Evaluate how smart the model got
# R-squared tells us the percentage of the data's story our model successfully understands
r2 = r2_score(y_test, predictions)
# Mean Absolute Error tells us how many dollars our guess is off by, on average
mae = mean_absolute_error(y_test, predictions)

print("--- Predictive Model Performance ---")
print(f"Model Accuracy (R-squared Score): {r2 * 100:.2f}%")
print(f"Average Prediction Error: ${mae:.2f}")

### Model Results & Interpretation

**R-squared (R²):** The model achieves an R² score of approximately **86%**. This means the model can explain 86% of the variation in insurance charges across customers — the remaining 14% is driven by factors not captured in this dataset (individual medical history, specific diagnoses, lifestyle factors beyond smoking).

An R² of 86% is considered **strong performance** for a regression model on real-world data. It is not perfect, but it is well above the threshold for practical usefulness.

**Mean Absolute Error (MAE):** The MAE of approximately **$2,500** means that on average, the model's charge prediction is within $2,500 of the actual charge. Given that charges range from roughly $1,100 to $63,770 — a spread of over $62,000 — an average error of $2,500 represents strong predictive precision.

**Why the model performs well:** The EDA and visualisation steps already revealed that smoking status is an extremely strong predictor of charges. When a feature has this much explanatory power, tree-based models like Random Forest capture it very effectively.

### Key Limitations
- The dataset contains only 1,337 records — a larger dataset would improve model stability
- The model does not have access to medical history, which in reality is a primary pricing factor
- Predictions should be treated as estimates, not clinical or financial guarantees

---

## Project Summary

| Finding | Detail |
|---------|--------|
| Strongest cost driver | Smoking status — smokers pay ~4x more on average |
| Age effect | Charges rise with age, but the effect is amplified dramatically by smoking |
| Model type | Random Forest Regressor |
| Model accuracy | ~86% R-squared on unseen test data |
| Average prediction error | ~$2,500 per customer |
| Records analysed | 1,337 (after removing 1 duplicate) |

**Conclusion:** This project demonstrates a complete data analytics pipeline — from raw data collection through to a working predictive model. The central finding is clear and actionable: smoking is not just a health risk, it is the dominant financial risk factor in medical insurance pricing, compounding with age to produce charges that are categorically higher than those of non-smokers across all age groups.